# LogiScan Phase 4: NaN Root Cause Investigation

This notebook performs a tensor-level audit to identify why training produced NaN loss.

### Instructions
1. Upload `unified_training_data.json` to Colab.
2. Run all cells.
3. Provide the output of the final cell back for analysis.

In [ ]:
!pip install -q transformers[torch] datasets sentencepiece

In [ ]:
import json

import numpy as np
import torch
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, AutoTokenizer

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def audit_everything():
    print("--- 1. AMP STATUS ---")
    print(f"Autocast Enabled: {torch.is_autocast_enabled()}")

    # Load Data
    with open("unified_training_data.json") as f:
        data = json.load(f)

    texts = [d["text"] for d in data]
    label_list = sorted(list(set(d["fallacy"] for d in data)))
    label2id = {l: i for i, l in enumerate(label_list)}
    labels = [label2id[d["fallacy"]] for d in data]

    print("\n--- 4-6. LABEL VALIDATION ---")
    print(f"Derived num_labels: {len(label_list)}")
    print(f"Label Mapping: {json.dumps(label2id, indent=2)}")
    ids = sorted(list(set(labels)))
    print(f"Contiguous (0 to N-1): {ids == list(range(len(ids)))}")

    print("\n--- 7. DATASET SCAN ---")
    nulls = sum(1 for t in texts if t is None)
    empties = sum(1 for t in texts if t is not None and not str(t).strip())
    print(f"Nulls: {nulls}, Empties: {empties}")

    print("\n--- 2-3. CLASS WEIGHTS ---")
    label_counts = np.bincount(labels)
    weights = 1.0 / (label_counts + 1e-6)
    weights = weights / weights.sum() * len(label_list)
    print(f"Exact Weights: {weights}")
    print(f"Weights check - NaN: {np.isnan(weights).any()}, Inf: {np.isinf(weights).any()}, Neg/Zero: {(weights <= 0).any()}")

    # Model Init
    MODEL_NAME = "microsoft/deberta-v3-small"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(label_list)).to(DEVICE)

    print("\n--- 6. WEIGHT INITIALIZATION AUDIT ---")
    for name, param in model.named_parameters():
        if "weight" in name and param.dim() > 1:
            std = param.std().item()
            max_val = param.abs().max().item()
            if std > 5.0 or max_val > 50.0:
                print(f"LAYER ALERT: {name} | std={std:.2f}, max={max_val:.2f}")

    print("\n--- 7-8. BATCH 0 FORWARD PASS INSTRUMENTATION ---")
    first_nan = {"found": False}

    def hook_fn(module, input, output):
        if first_nan["found"]: return
        t = output[0] if isinstance(output, tuple) else output
        if isinstance(t, torch.Tensor):
            if torch.isnan(t).any() or torch.isinf(t).any():
                first_nan["found"] = True
                print(f"!!! FIRST INF/NAN DETECTED IN MODULE: {module._logiscan_name} !!!")
                print(f"Type: {type(module)}")
                print(f"Shape: {t.shape}")
                print(f"Min: {t.min().item()}, Max: {t.max().item()}")
                if torch.isinf(t).any():
                    idx = torch.nonzero(torch.isinf(t))[0].tolist()
                    print(f"First Inf Index: {idx}")

    for name, module in model.named_modules():
        module._logiscan_name = name
        module.register_forward_hook(hook_fn)

    batch_texts = texts[:4]
    batch_labels = torch.tensor(labels[:4]).to(DEVICE)
    inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True).to(DEVICE)

    print(f"Tokenizer NaNs: {torch.isnan(inputs.input_ids.float()).any().item()}")

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        print(f"Logits Min/Max: {logits.min().item():.4f} / {logits.max().item():.4f}")

        loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float).to(DEVICE))
        loss = loss_fn(logits, batch_labels)
        print(f"Batch 0 Loss: {loss.item()}")
        print(f"Label IDs: {batch_labels.tolist()}")

audit_everything()